In [66]:
import pandas as pd
import networkx as nx
import community as community_louvain  # pip install python-louvain
from pathlib import Path

# ===============================
# 1️⃣ 构建加权无向三层网络
# ===============================
def build_weighted_multilayer_network(df, zone_col='zone', host_col='host',virus_col='virus'):
    G = nx.Graph()

    # zone–host 权重
    zh_weight = df.groupby([zone_col, host_col]).size()
    # host–virus 权重
    hv_weight = df.groupby([host_col, virus_col]).size()

    # 添加节点
    for z in df[zone_col].unique():
        G.add_node(z, layer='zone')
    for h in df[host_col].unique():
        G.add_node(h, layer='host')
    for v in df[virus_col].unique():
        G.add_node(v, layer='virus')

    # 添加加权边
    for (z, h), w in zh_weight.items():
        G.add_edge(z, h, weight=w)

    for (h, v), w in hv_weight.items():
        G.add_edge(h, v, weight=w)

    return G


# ===============================
# 2️⃣ 计算整体模块度 Q
# ===============================
def compute_overall_modularity(G, random_state=42):
    partition = community_louvain.best_partition(
        G,
        weight='weight',
        random_state=random_state
    )

    Q = community_louvain.modularity(
        partition,
        G,
        weight='weight'
    )

    return Q, partition


# ===============================
# 3️⃣ 主程序
# ===============================
import pandas as pd
from pathlib import Path

def analyze_network_by_zone():
    # ---- 读取并整理数据 ----
    # df = pd.read_excel(r'input/host_virus-NEW-0313.xlsx')
    # df = df[df['virus_new'].notna()].reset_index(drop=True)
    # df = df[['zone', 'Host_species', 'virus_new', 'County']]
    # df = df.rename(columns={'Host_species': 'host', 'virus_new': 'virus'})

    df = pd.read_excel(r'input/host_virus-XL-Zone-BoundaryMatch_20260112.xlsx')
    df = df[df['virus'].notna()].reset_index(drop=True)
    df = df[['边界线匹配后的分区', 'Host_species', 'virus', 'County']]
    df = df.rename(columns={'Host_species': 'host', '边界线匹配后的分区': 'zone'})

    df_new = (
        df
        .assign(virus=df['virus'].str.split(','))
        .explode('virus')
        .assign(virus=lambda x: x['virus'].str.strip())
        .reset_index(drop=True)
    )
    df_new[['host', 'virus']] = df_new[['host', 'virus']].apply(lambda x: x.str.strip())
    # df_new.to_excel(r'input/data_split.xlsx', index=False)
    # ---- 构建网络 ----
    G = build_weighted_multilayer_network(df_new)

    print(f'节点数: {G.number_of_nodes()}')
    print(f'边数: {G.number_of_edges()}')

    # ---- 计算整体模块度 ----
    Q, partition = compute_overall_modularity(G, random_state=1)

    print('\n===== 整体网络模块度 =====')
    print(f'Modularity Q = {Q:.4f}')

    # ---- 保存结果 ----
    # outdir = Path(output_dir)
    # outdir.mkdir(parents=True, exist_ok=True)

    # 文件后缀根据变量名生成（去掉空格和特殊字符）
    partition_df = pd.DataFrame([
        {
            'node': node,
            'layer': G.nodes[node].get('layer', 'unknown'),
            'module': module
        }
        for node, module in partition.items()
    ])

    partition_df.to_excel(
        f'output/3_modularity/network_partition_old.xlsx',
        index=False
    )

    with open(f'output/3_modularity/modularity_Q_old.txt', 'w') as f:
        f.write(f'Modularity Q = {Q:.4f}\n')

    return G, Q, partition, partition_df


# ===============================
# 1️⃣ 示例引用
# ===============================
if __name__ == "__main__":
    G, Q, partition, partition_df = analyze_network_by_zone()

节点数: 2231
边数: 3215

===== 整体网络模块度 =====
Modularity Q = 0.5536


In [67]:
df1=pd.read_excel(r'output/3_modularity/network_partition_old.xlsx')

In [69]:
result = (
    df1.groupby(['module', 'layer'])['node']
      .nunique()
      .unstack(fill_value=0)
      .reset_index()
)

print(result)

layer  module  host  virus  zone
0           0    36    925     1
1           1    11    119     1
2           2    14    202     1
3           3    19    603     1
4           4     1      3     0
5           5     2      3     0
6           6     1    198     0
7           7     6     47     0
8           8     2      3     0
9           9     1      4     0
10         10     1      4     0
11         11     1      1     0
12         12     1      2     0
13         13     1     16     0


In [71]:
df = pd.read_excel(r'input/host_virus-XL-Zone-BoundaryMatch_20260112.xlsx')
df = df[df['virus'].notna()].reset_index(drop=True)
df = df[['边界线匹配后的分区', 'Host_species', 'virus', 'County']]
df = df.rename(columns={'Host_species': 'host', '边界线匹配后的分区': 'zone'})

# 拆分多病毒行
df_new = (
    df
    .assign(virus=df['virus'].str.split(','))
    .explode('virus')
    .assign(virus=lambda x: x['virus'].str.strip())
    .reset_index(drop=True)
)
df_new[['host', 'virus']] = df_new[['host', 'virus']].apply(lambda x: x.str.strip())

In [73]:
for zone in [1, 2, 3, 4]:
    tmp=len(df_new[df_new['zone']==zone]['host'].unique())
    print(zone, tmp)

1 64
2 27
3 13
4 16


In [ ]:
hosts=df1[(df1['module']==3) & (df1['layer']=='host')]['node'].unique()
print(len(hosts))
tmp=len(df_new[df_new['host'].isin(hosts)]['virus'].unique())
print(tmp)

19
